# Phase A: Download TOXRIC and inspect the schema

This notebook covers your **Steps 1 and 2**:
1. Download the real TOXRIC dataset (from Figshare DOI 27195339)
2. Show the dataset schema (columns, dtypes, SMILES column, target columns)

It also clones the `lihan97/KPGT` repo so we can use its real featurizer in Phase B.

All download URLs and KPGT class names in this project were verified against the actual Figshare API and GitHub source — no placeholders.

**Before running:** `pip install -r requirements.txt`


## Step 1 — Download TOXRIC

TOXRIC's web portal uses JS-driven downloads that bots can't trigger directly, so we pull the archive directly from the authors' Figshare record. The 30-datasets archive (~7.7 MB) contains 30 per-endpoint CSVs covering classification and regression toxicity tasks.


In [ ]:
import sys
from pathlib import Path

# Make src importable from inside the notebook
sys.path.insert(0, str(Path.cwd()))

from toxpkg.data_utils import download_toxric, list_toxric_subdatasets

extract_dir = download_toxric(which="toxric_30_datasets.zip")
csvs = list_toxric_subdatasets(extract_dir)
print(f"\nExtracted {len(csvs)} CSV files. First 5:")
for p in csvs[:5]:
    print(f"  {p.relative_to(extract_dir)}")


## Step 2 — Inspect schema of a TOXRIC sub-dataset

Each TOXRIC sub-dataset is a flat CSV. The schema we expect (and verify here):

- **Compound identifier**: `TAID` (TOXRIC's internal compound ID)
- **Input column**: a SMILES string (column name varies — `Canonical SMILES`, `smiles`, etc.)
- **Target column(s)**: one or more toxicity endpoints (binary hit/no-hit, or continuous like log LD50)

Pick any CSV from the list above and inspect it. `inspect_csv` returns columns, dtypes, a sample of rows, and a best guess at which column is SMILES.


In [ ]:
from toxpkg.data_utils import inspect_csv, print_inspection

# Inspect the first CSV. Change the index to look at others.
info = inspect_csv(csvs[0])
print_inspection(info)

print("\n--- Summary across all sub-datasets ---")
for csv in csvs:
    quick = inspect_csv(csv, n_rows=0)
    print(f"  {csv.name:<55} rows={quick['n_rows_total']:>6}  cols={len(quick['columns'])}")


## Step 3 — Set up KPGT (for Phase B)

Clones the real `lihan97/KPGT` repo into `external/KPGT/` so we can reuse the authors' featurizer (`smiles_to_graph_tune`), the `LiGhTPredictor` model class, and the `Collator_tune` for batching.

**Note:** the pretrained `base.pth` checkpoint lives behind an anonymous Figshare share link with no public API. The function below tells you the exact URL to download from and where to put the file. This is a one-time manual step.


In [ ]:
from toxpkg.data_utils import clone_kpgt, check_kpgt_pretrained

kpgt_dir = clone_kpgt()
check_kpgt_pretrained(kpgt_dir)


## What's next (Phase B)

Once you've inspected the schema and know which sub-dataset(s) you want to fine-tune on, Phase B will add:

- `src/featurizer.py` — wrap KPGT's `smiles_to_graph_tune` to turn SMILES into DGL graphs
- `src/dataset.py` — convert a TOXRIC CSV into the KPGT-cached format (`{name}_5.pkl`, `rdkfp1-7_512.npz`, `molecular_descriptors.npz`) by reusing KPGT's `scripts/preprocess_downstream_dataset.py`
- `src/model.py` — instantiate the real `LiGhTPredictor` with the base config (`d_g_feats=768, n_mol_layers=12, n_heads=12`) and load `base.pth`
- `src/train.py` — masked-loss training loop using KPGT's `Trainer` from `src/trainer/finetune_trainer.py`

Everything in Phase B will use class names and signatures verified from the actual repo source — not invented ones.
